# MAGFiLO / Solar Filament — **FINAL V25 (public 0.39) Paper & Component-Ablation Laboratory**

**Primary target: the existing V25 0.39-public-score pipeline—not a new leaderboard submission.** This notebook **runs real V25 components**, performs **same-scene** removal/addition experiments, three-seed ranker feature-family refits, instance-size and annotator analyses, per-day paired uncertainty, failure visualizations, inference timing, and exports CSV/LaTeX/PNG/PDF + an integrity manifest. Its exact historical V25 scored CSV is preserved; a reconstructed V25 output is **never** labeled public 0.39 without passing hash/instance agreement checks.

**Required Kaggle inputs**: official MAGFiLO `solar3k` dataset; trained V20 `portable_global_m_best.pt` and `portable_native_m_best.pt`; V24 `SOLAR_V24_REFINER_BEST.pt`, `SOLAR_V24_INSTANCE_QUALITY.joblib`; V25 `SOLAR_V25_CURVE_BEST.pt`, `SOLAR_V25_REAL_REFINER_BEST.pt`, `SOLAR_V25_EXTENDED_QUALITY.joblib`; **original** `V25_REAL_REFINE_PLUS_QUALITY.csv` inside V25 full prediction/report bundle or directly. V25 files can be attached using the working `lighittha/v25ftss` Kaggle dataset shown in your earlier run logs.

**Evidence scope:** 707 labeled train scenes and 180 hidden-label test scenes. The historical V20 104-scene holdout was repeatedly used for model selection during V20–V29 and **cannot** support claims of independent generalization or unbiased p-values. The 53 CAL / 51 AUDIT scenes remain **historical** development/audit slices. Official organizer PQ parity is NOT established by this proxy. Public 0.39 applies only to the exact historical scored CSV; local PQ is distinct.

**One run:** Kaggle GPU T4 (Internet ON if dependencies missing). Run All. The notebook can be lengthy because it uses true 2048px/1792px image inference and learned component loading. It writes durable stage-level cache files. Optional original-test reproduction is off by default because the original scored CSV is already available; change `RUN_TEST_REPRODUCTION=True` to compare all 180 regenerated scenes with the original file (without test labels).

**Output:** `/kaggle/working/SOLAR_V25_FINAL_PAPER_EVIDENCE.zip` plus working folder `/kaggle/working/solar_v25_final_paper/`. A saved notebook version makes output persistent. No use of hidden test ground-truth annotations.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input');OUT=Path('/kaggle/working/solar_v25_final_paper')
SEED=2026;GPU_ID=0;H=W=2048
RAW_CONF=.08;RAW_IOU=.72;MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False;REQUIRE_V24_MODELS=True;FULL_VALIDATION_REQUIRED=True
V25_EXPECTED_QUALITY_CUTOFF=.38
V25_EXPECTED_CSV_SHA='ac470fbfa0e2ca5f2d'; V25_EXPECTED_ROWS=1093
CROP_SIDE=320
CURVE_WINDOW=768;CURVE_STRIDE=640;CURVE_SIDE=512;CURVE_MIN_AREA=170;CURVE_MAX_PROPOSALS=42
# V29: independently ranked recovery candidates, not retraining a baseline that already works.
LOW_GLOBAL=.115;LOW_TILE=.145;LOW_CURVE=.69;MIN_EXTRA_AREA=180
MAX_GLOBAL_RAW=26;MAX_TILE_RAW=42;MAX_CURVE_RAW=8
MAX_SINGLES_PER_SCENE=28;MAX_FRAGMENT_PAIRS=9
PAIR_END_DISTANCE=90.; PAIR_MAX_IOU=.12
MAX_NEW_ACCEPTED_PER_SCENE=4
MAX_ACCEPTED_OVERLAP=.22; MIN_FREE_FRACTION=.78
CANDIDATE_THRESHOLDS=[.035,.055,.075,.095,.125,.16,.20,.24,.28]
MIN_PROMOTION_AUDIT_GAIN=.004;MIN_BOOTSTRAP_POSITIVE=.90
SCORER_TRAINING_SPLIT='historical_calibration_53_scenes';TEST_LABELS_FORBIDDEN=True
for d in ('models','raw','refined','curve_raw','reports','submissions','input_data','candidate_cache','feature_cache','labels','checkpoints'):
    (OUT/d).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(),'Run on Kaggle with solar3k and V25 assets.'
print('V29 output:',OUT)

In [ ]:
from pathlib import Path
OUT=Path('/kaggle/working/solar_v25_final_paper')
for folder in ('tables','figures','evidence','cache','models','reports','raw','refined','curve_raw','input_data'): (OUT/folder).mkdir(parents=True,exist_ok=True)
RUN_RANKER_FEATURE_ABLATIONS=True
RANKER_SEEDS=(2026,2037,2048)
RUN_TEST_REPRODUCTION=False
RUN_ERROR_VISUALIZATIONS=True
VISUALIZATION_CASES=8
BOOTSTRAP_REPLICATES=2000
assert V25_EXPECTED_CSV_SHA=='ac470fbfa0e2ca5f2d' and V25_EXPECTED_ROWS==1093

## 1 · Environment, original dataset, model integrity

In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.39,'target_public_PQ':0.40})

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','v25','v26','model','trained','prediction','report')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

In [ ]:
# Restore all genuinely trained V25 checkpoint bytes; no substitute/pretrained-only networks.
from skimage.morphology import skeletonize
from skimage.segmentation import watershed

V25_REAL_PATH=OUT/'models'/'SOLAR_V25_REAL_REFINER_BEST.pt'
V25_CURVE_PATH=OUT/'models'/'SOLAR_V25_CURVE_BEST.pt'
V25_QUALITY_PATH=OUT/'models'/'SOLAR_V25_EXTENDED_QUALITY.joblib'
for p in (V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
    data,source=find_original_asset(p.name)
    if data is None:raise FileNotFoundError('Missing '+p.name+'. Attach the complete V25 trained checkpoint ZIP.')
    if not p.exists() or sha(p)!=hashlib.sha256(data).hexdigest():p.write_bytes(data)
    print('Recovered frozen V25',p.name,sha(p)[:18], 'from',source)
refiner_real=PromptUNet().cuda(GPU_ID)
refiner_real.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_real.eval();REAL_REFINER_SHA=sha(V25_REAL_PATH)

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

v25_ranker_package=joblib.load(V25_QUALITY_PATH)
new_rankers=v25_ranker_package['models']
V25_QUALITY_CUTOFF=float(v25_ranker_package['cutoff'])
print('Frozen V25 ranker cutoff:',V25_QUALITY_CUTOFF,'feature count:',len(v25_ranker_package['columns']))
assert abs(V25_QUALITY_CUTOFF-V25_EXPECTED_QUALITY_CUTOFF)<1e-6, 'Wrong V25 ranking checkpoint/cutoff'

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([m.predict_proba(a)[:,1] for m in models],axis=0)

In [ ]:
class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)


curve_path=V25_CURVE_PATH
curve_net=MiniUNet().cuda(GPU_ID)
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'],strict=True)
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Loaded frozen V25 curve network',CURVE_SHA[:18])

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out


assert list(v25_ranker_package['columns'])==NEW_FEATURES,'V25 feature order mismatch'

In [ ]:
# Use all frozen V25 components to reconstruct its exact candidate-order/quality behavior.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    curve_rles=[as_rle(p['counts']) for p in curve_list]
    meta=[]
    for i,rr in enumerate(full):
        if origins[i]=='detector':meta.append({});continue
        overlaps=np.asarray(coco.iou([rr],curve_rles,[0]*len(curve_rles)),float).reshape(-1) if curve_rles else np.asarray([])
        meta.append(curve_list[int(overlaps.argmax())] if len(overlaps) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    location=OUT/'reports'/('v25_features_'+stem+'.joblib')
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+sha(V25_QUALITY_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if location.is_file():
        try:
            obj=joblib.load(location)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    result={'signature':signature,'masks':[txt(r) for r in masks], 'origins':origins,
            'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(result,location,compress=3)
    return result

def nonoverlap_after_refinement(masks):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in masks:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

def frozen_v25_masks(stem, obj=None):
    info=obj if obj is not None else scene_feature_data(stem)
    prob=rank_probs(new_rankers,info['features'])
    inds=[i for i in range(info['n_base']) if prob[i]>=V25_QUALITY_CUTOFF]
    masks=[as_rle(info['masks'][i]) for i in inds]
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=[real_refine(stem,rr,gray) if prob[i]>=.50 else rr for i,rr in zip(inds,masks)]
    return nonoverlap_after_refinement(refined)

assert V25_QUALITY_CUTOFF==V25_EXPECTED_QUALITY_CUTOFF
assert list(v25_ranker_package['columns'])==NEW_FEATURES
print('V25 0.39 candidate logic restored, original ranker/curve/refiner frozen.')

## 2 · Historical public 0.39 evidence gate

The source notebook explicitly documented the **original scored CSV** fingerprint `ac470fbfa0e2ca5f2d...` and 1,093 instances. The file, not a regenerated approximation, is the public reference. This cell fails clearly if it is absent; the remaining component study should not misrepresent a reconstructed file as the original scored submission.

In [ ]:
import csv
PROVEN=OUT/'evidence'/'V25_EXACT_PUBLIC_039.csv'
original_found=False
for filename in ('V25_REAL_REFINE_PLUS_QUALITY.csv','submission.csv'):
    payload,asset_source=find_original_asset(filename)
    if payload is None:continue
    if hashlib.sha256(payload).hexdigest().startswith(V25_EXPECTED_CSV_SHA):
        PROVEN.write_bytes(payload);original_found=True;break
if not original_found:
    raise FileNotFoundError('The exact V25 0.39 CSV is required. Attach V25_REAL_REFINE_PLUS_QUALITY.csv or SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip (the trained checkpoint ZIP alone is insufficient).')
assert sha(PROVEN).startswith(V25_EXPECTED_CSV_SHA)
control_frame=pd.read_csv(PROVEN,dtype=str)
assert control_frame.columns.tolist()==['filament_id','segmentation_rle']
assert len(control_frame)==V25_EXPECTED_ROWS and control_frame.filament_id.is_unique
CONTROL_BY_STEM=defaultdict(list)
for row in control_frame.itertuples(index=False):
    stem=row.filament_id.rsplit('_',1)[0]
    assert stem in TEST
    CONTROL_BY_STEM[stem].append(as_rle(row.segmentation_rle))
assert set(CONTROL_BY_STEM).issubset(TEST)
save_json(OUT/'evidence'/'historical_public_provenance.json',dict(
    confirmed_public_score=.39, public_score_source='historical user-reported Kaggle V25 public leaderboard',
    original_csv_sha256=sha(PROVEN),original_csv_source=asset_source,
    instance_rows=len(control_frame),test_scene_count=len(TEST),
    reconstructed_csv_is_not_public_scored=True,test_labels_used=False))
print('Verified exact original V25 public 0.39 CSV fingerprint:',sha(PROVEN)[:24])

## 3 · Define actual V25 component-removal experiments

All ablations use identical source images, fixed trained weights, and common evaluation code. The historical V25 winning control uses the original **detector-only scored candidate list**, but the curve branch can affect neighborhood/context features of detector candidates. Therefore `remove_curve_context` recalculates detector candidate features without curve proposals; `enable_curve_detections` instead tests a true candidate-inclusion intervention. No feature is replaced with ground-truth metadata.

For *frozen* feature groups, zeroing a feature is a distribution shift, not retraining. The notebook therefore uses a **separate ranker-retraining experiment** below for legitimate feature-family ablations. All component comparisons remain historical/audit exploratory.

In [ ]:
from time import perf_counter
from collections import OrderedDict
from scipy.optimize import linear_sum_assignment

FROZEN_METHODS=OrderedDict([
 ('D0_global_only','global'),
 ('D1_native_only','native'),
 ('D2_two_detector_fusion','detector'),
 ('D3_v24_quality_gate','v24'),
 ('V25_no_curve_context','no_curve'),
 ('V25_no_quality_ranker','no_ranker'),
 ('V25_no_real_mask_refinement','no_real_refiner'),
 ('V25_full_reconstructed','full'),
 ('V25_add_curve_candidates','add_curve'),
 ('V25_always_refine_selected','always_refine'),
 ('V25_ranker_cutoff_030','threshold_030'),
 ('V25_ranker_cutoff_046','threshold_046'),
])

FEATURE_GROUPS={
 'shape_topology': ['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
    'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count',
    'skeleton_length','skeleton_area_ratio','mean_thickness','max_thickness','width_variation',
    'local_component_count','hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos'],
 'image_appearance': ['pixel_avg_inside','pixel_avg_ring','contrast','intensity_inside_std',
    'intensity_inside_p10','intensity_inside_p50','intensity_inside_p90','intensity_ring_p50',
    'intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
    'sobel_ring_mean','contrast_normalized','intensity_outside_std'],
 'detector_support': ['base_score','global_agreement','tile_agreement','global_maxconf',
    'tile_maxconf','curve_vs_global_iou','curve_vs_tile_iou','curve_conf','curve_center_support',
    'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
    'mask_global_rank','mask_tile_rank','curve_origin'],
 'neighbor_context': ['neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
    'sun_radial','rim_distance','mask_near_solar_limb','base_center_x','base_center_y','mask_area_over_disk'],
}
assert set().union(*map(set,FEATURE_GROUPS.values())).issubset(set(NEW_FEATURES))

def ranked_selection(stem,info,cutoff=V25_QUALITY_CUTOFF,use_real=True,all_refine=False,
                     include_curves=False):
    prob=rank_probs(new_rankers,info['features'])
    last=len(info['masks']) if include_curves else info['n_base']
    inds=[i for i in range(last) if float(prob[i])>=cutoff]
    masks=[as_rle(info['masks'][i]) for i in inds]
    if not use_real:return nonoverlap_after_refinement(masks)
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    masks=[real_refine(stem,mask,gray) if (all_refine or float(prob[i])>=.50) and
           (i<info['n_base']) else mask for i,mask in zip(inds,masks)]
    return nonoverlap_after_refinement(masks)

def no_curve_info(stem):
    raw,base=load_base(stem)
    features=extended_features(stem,base,origins=['detector']*len(base),
                               curve_meta=[{} for _ in base],raw=raw)
    return dict(masks=[txt(rr) for rr in base],features=features,n_base=len(base))

def scene_variants(stem,methods=None):
    methods=set(FROZEN_METHODS if methods is None else methods)
    info=scene_feature_data(stem)
    raw,base=load_base(stem)
    assert info['n_base']==len(base)
    results={}
    if 'D0_global_only' in methods:
        c=dict(WINNER,sources=[list(SRC_HI)]);results['D0_global_only']=select_masks(raw,c)
    if 'D1_native_only' in methods:
        c=dict(WINNER,sources=[list(SRC_TILES)]);results['D1_native_only']=select_masks(raw,c)
    if 'D2_two_detector_fusion' in methods:results['D2_two_detector_fusion']=base
    if 'D3_v24_quality_gate' in methods:results['D3_v24_quality_gate']=v24_control(stem)
    if 'V25_no_curve_context' in methods:
        results['V25_no_curve_context']=ranked_selection(stem,no_curve_info(stem))
    if 'V25_no_quality_ranker' in methods:
        gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
        results['V25_no_quality_ranker']=nonoverlap_after_refinement([real_refine(stem,r,gray) for r in base])
    if 'V25_no_real_mask_refinement' in methods:
        results['V25_no_real_mask_refinement']=ranked_selection(stem,info,use_real=False)
    if 'V25_full_reconstructed' in methods:
        results['V25_full_reconstructed']=frozen_v25_masks(stem,info)
    if 'V25_add_curve_candidates' in methods:
        results['V25_add_curve_candidates']=ranked_selection(stem,info,include_curves=True)
    if 'V25_always_refine_selected' in methods:
        results['V25_always_refine_selected']=ranked_selection(stem,info,all_refine=True)
    if 'V25_ranker_cutoff_030' in methods:
        results['V25_ranker_cutoff_030']=ranked_selection(stem,info,cutoff=.30)
    if 'V25_ranker_cutoff_046' in methods:
        results['V25_ranker_cutoff_046']=ranked_selection(stem,info,cutoff=.46)
    assert set(results)==methods,(set(results),methods)
    return results,info

assert abs(V25_QUALITY_CUTOFF-.38)<1e-6
print('Real component variants:',len(FROZEN_METHODS),'| ranker features:',len(NEW_FEATURES))

## 4 · Paired historical audit: PQ, SQ, RQ, Dice, TP/FP/FN, per-scene exports

In [ ]:
# The score function physical_counts() is the historical per-expert instance-PQ proxy.
# All methods are compared on the same labeled acquisition-date groups.
def foreground_union(rles):
    result=np.zeros((H,W),np.uint8)
    if rles:
        mask=coco.decode(rles)
        result=np.any(mask>0,axis=2).astype(np.uint8) if mask.ndim==3 else (mask>0).astype(np.uint8)
    return result

def mean_expert_dice(stem,predicted):
    union_pred=foreground_union(predicted)
    scores=[]
    for truth in gt(stem):
        mask=foreground_union(truth)
        den=int(union_pred.sum())+int(mask.sum())
        scores.append(2*int(np.logical_and(union_pred,mask).sum())/den if den else 1.)
    return float(np.mean(scores)) if scores else np.nan

torch.cuda.reset_peak_memory_stats(GPU_ID)
EVALUATION_STEMS=sorted(CAL+AUDIT)
assert len(EVALUATION_STEMS)==104 and len(CAL)==53 and len(AUDIT)==51
PER_SCENE=[];PER_VERSION=[];TIMINGS=[];CONTROL_PREDICTIONS={}
PROGRESS=OUT/'evidence'/'ablation_progress.jsonl'
if PROGRESS.exists():PROGRESS.unlink()
for stem in tqdm(EVALUATION_STEMS,desc='FINAL frozen-model component ablations'):
    t0=perf_counter()
    variants,info=scene_variants(stem)
    elapsed=perf_counter()-t0
    TIMINGS.append(dict(stem=stem,seconds_all_variants=elapsed,model_count=len(variants)))
    if stem in AUDIT:CONTROL_PREDICTIONS[stem]=[txt(r) for r in variants['V25_full_reconstructed']]
    for method,pred in variants.items():
        counts=physical_counts(stem,pred)
        row=dict(stem=stem,day=stem[:8],partition='CAL_historical' if stem in CAL else 'AUDIT_historical',
                 method=method,sum_iou=float(counts[0]),TP=float(counts[1]),FP=float(counts[2]),FN=float(counts[3]),
                 instances=len(pred),dice=mean_expert_dice(stem,pred))
        PER_SCENE.append(row)
    with open(PROGRESS,'a',encoding='utf-8') as log:
        log.write(json.dumps({'stem':stem,'elapsed_s':round(elapsed,3),'done':True})+'\n')
    if stem in AUDIT:
        for method,pred in variants.items():
            for k,truth in enumerate(gt(stem)):
                c=match_counts(truth,pred)
                PER_VERSION.append(dict(stem=stem,day=stem[:8],method=method,expert_version=k,
                                        sum_iou=float(c[0]),TP=int(c[1]),FP=int(c[2]),FN=int(c[3])))
per_scene=pd.DataFrame(PER_SCENE)
per_scene.to_csv(OUT/'tables'/'Table_scene_level_all_component_ablations.csv',index=False)
pd.DataFrame(PER_VERSION).to_csv(OUT/'tables'/'Table_expert_version_counts.csv',index=False)
pd.DataFrame(TIMINGS).to_csv(OUT/'tables'/'Table_pipeline_runtime_seconds.csv',index=False)

def compact_metrics(frame):
    counts=frame[['sum_iou','TP','FP','FN']].sum().to_numpy(float)
    result=metrics(counts)
    result.update(dice=float(frame['dice'].mean()),scenes=int(frame.stem.nunique()),days=int(frame.day.nunique()),
                  instances=int(frame.instances.sum()))
    return result

TABLES=[]
for (part,method),group in per_scene.groupby(['partition','method']):
    TABLES.append(dict(partition=part,method=method,**compact_metrics(group)))
summary=pd.DataFrame(TABLES)
ctrl=summary[(summary.partition=='AUDIT_historical')&(summary.method=='V25_full_reconstructed')].PQ.iloc[0]
summary['delta_vs_reconstructed_V25']=np.where(summary.partition=='AUDIT_historical',summary.PQ-ctrl,np.nan)
summary=summary.sort_values(['partition','PQ'],ascending=[True,False])
summary.to_csv(OUT/'tables'/'Table01_same_scene_component_ablation.csv',index=False)
(OUT/'tables'/'Table01_same_scene_component_ablation.tex').write_text(summary.to_latex(index=False,float_format='%.4f'))
display(summary[summary.partition=='AUDIT_historical'])
assert set(per_scene.method)==set(FROZEN_METHODS)
print('Peak GPU allocation during study:',round(torch.cuda.max_memory_allocated(GPU_ID)/2**30,2),'GiB')

## 5 · Paired acquisition-day bootstrap uncertainty (historical selection bias disclosed)

In [ ]:
# Historical development audit; bootstrap is descriptive only (not confirmatory hypothesis test).
audit_scene=per_scene[per_scene.partition=='AUDIT_historical']
per_day=audit_scene.groupby(['method','day'],as_index=False)[['sum_iou','TP','FP','FN']].sum()
DAYS=sorted(per_day.day.unique())
assert len(DAYS)>=35
vectors={name:np.array([per_day[(per_day.method==name)&(per_day.day==d)]
                               [['sum_iou','TP','FP','FN']].iloc[0].to_numpy(float) for d in DAYS])
         for name in FROZEN_METHODS}
reps=np.random.default_rng(SEED+2025).integers(0,len(DAYS),size=(BOOTSTRAP_REPLICATES,len(DAYS)))

def pq_batch(count):
    den=count[...,1]+.5*(count[...,2]+count[...,3])
    return np.divide(count[...,0],den,out=np.zeros_like(den,dtype=float),where=den>0)
ref_boot=pq_batch(vectors['V25_full_reconstructed'][reps].sum(axis=1))
BOOT=[]
for method,vec in vectors.items():
    trial=pq_batch(vec[reps].sum(axis=1))
    change=trial-ref_boot
    BOOT.append(dict(method=method,PQ=pq_batch(vec.sum(axis=0)),
                     delta_vs_full=float(np.mean(change)),
                     lower_95=float(np.quantile(change,.025)),upper_95=float(np.quantile(change,.975)),
                     bootstrap_positive_fraction=float(np.mean(change>0)),
                     interpretation='historical reused audit; CI descriptive only'))
bootstrap=pd.DataFrame(BOOT).sort_values('PQ',ascending=False)
bootstrap.to_csv(OUT/'tables'/'Table02_paired_day_bootstrap.csv',index=False)
(OUT/'tables'/'Table02_paired_day_bootstrap.tex').write_text(bootstrap.to_latex(index=False,float_format='%.4f'))
display(bootstrap)

## 6 · Expert agreement, filament-size strata, and fragmentation/error morphology

In [ ]:
from skimage.morphology import skeletonize
agreement=[]; size_rows=[]; error_rows=[]
SIZE_BINS=[(0,1000,'tiny_<1k'),(1000,5000,'small_1k_5k'),(5000,20000,'medium_5k_20k'),(20000,float('inf'),'large_20k_plus')]
for stem in tqdm(AUDIT,desc='Expert agreement and filament size recall'):
    truths=gt(stem)
    for i in range(len(truths)):
        for j in range(i+1,len(truths)):
            c=match_counts(truths[i],truths[j])
            agreement.append(dict(stem=stem,day=stem[:8],expert_i=i,expert_j=j,
                                  PQ=pq(c),sum_iou=float(c[0]),TP=int(c[1]),FP=int(c[2]),FN=int(c[3])))
    pred=[as_rle(s) for s in CONTROL_PREDICTIONS[stem]]
    for version,truth in enumerate(truths):
        if not truth:continue
        ious=np.asarray(coco.iou(truth,pred,[0]*len(pred)),float) if pred else np.zeros((len(truth),0))
        matched={}
        if len(pred):
            ii,jj=linear_sum_assignment(-ious)
            matched={int(i):float(ious[i,j]) for i,j in zip(ii,jj) if ious[i,j]>.5}
        for k,rr in enumerate(truth):
            area=int(coco.area(rr));box=coco.toBbox(rr)
            bw,bh=float(box[2]),float(box[3])
            label=next(name for lower,upper,name in SIZE_BINS if lower<=area<upper)
            size_rows.append(dict(stem=stem,day=stem[:8],version=version,area=area,
                             bbox_short_side=min(bw,bh),bbox_elongation=max(bw,bh)/max(1,min(bw,bh)),
                             group=label,matched=int(k in matched),match_iou=matched.get(k,0)))
    # Count failure modes against each expert separately.
    for version,truth in enumerate(truths):
        pred_iou=np.asarray(coco.iou(truth,pred,[0]*len(pred)),float) if truth and pred else np.zeros((len(truth),len(pred)))
        n_split=sum((pred_iou[i]>.1).sum()>=2 for i in range(len(truth)))
        n_merge=sum((pred_iou[:,j]>.1).sum()>=2 for j in range(len(pred)))
        error_rows.append(dict(stem=stem,day=stem[:8],expert_version=version,
            possible_one_to_many=int(n_split),possible_many_to_one=int(n_merge),
            threshold_note='overlap >0.1; heuristic, not confirmed physical fragmentation'))
agreement_df=pd.DataFrame(agreement)
size_df=pd.DataFrame(size_rows)
errors_df=pd.DataFrame(error_rows)
for filename,frame in [('Table03_inter_annotator_agreement.csv',agreement_df),
   ('Table04_instance_size_recall.csv',size_df),('Table05_overlap_multiplicity.csv',errors_df)]:
    frame.to_csv(OUT/'tables'/filename,index=False)
size_summary=size_df.groupby('group',observed=False).agg(objects=('matched','size'),
    recall=('matched','mean'),mean_matched_iou=('match_iou','mean')).reset_index()
size_summary.to_csv(OUT/'tables'/'Table04_size_recall_summary.csv',index=False)
print('Expert pairs:',len(agreement_df),'mean pair PQ:',agreement_df.PQ.mean() if len(agreement_df) else None)
display(size_summary)

## 7 · Genuine held-audit ranker ablations: retrain feature families, repeat across seeds

In [ ]:
# Frozen V25 checkpoint remains unchanged. New quality models are trained ONLY on historical CAL,
# evaluated on the 51 historical audit scenes, and never promoted to the historical 0.39 label.
from sklearn.ensemble import HistGradientBoostingClassifier,ExtraTreesClassifier
from sklearn.metrics import roc_auc_score,brier_score_loss

feature_sets={'FULL_69':NEW_FEATURES}
for family,names in FEATURE_GROUPS.items():
    feature_sets['WITHOUT_'+family.upper()]=[f for f in NEW_FEATURES if f not in names]

CAL_BANK={};AUDIT_BANK={};FEATURE_RESULTS=[];FEATURE_PRED=[]
if RUN_RANKER_FEATURE_ABLATIONS:
    for stem in tqdm(CAL,desc='Create ranker ablation CAL examples'):
        info=scene_feature_data(stem)
        mask=[as_rle(s) for s in info['masks']]
        CAL_BANK[stem]=(info,consensus_quality_labels(stem,mask))
    for stem in tqdm(AUDIT,desc='Cache AUDIT features, labels only for evaluation'):
        info=scene_feature_data(stem)
        AUDIT_BANK[stem]=info
    X_train=pd.concat([CAL_BANK[s][0]['features'] for s in CAL],ignore_index=True)
    y_train=np.asarray([z for s in CAL for z in CAL_BANK[s][1]],int)
    assert len(X_train)==len(y_train) and len(np.unique(y_train))==2
    for seed in RANKER_SEEDS:
        for name,fields in tqdm(feature_sets.items(),desc=f'Quality ranker retrain seed {seed}'):
            models=[HistGradientBoostingClassifier(max_iter=140,max_leaf_nodes=11,min_samples_leaf=16,
                     learning_rate=.045,l2_regularization=7.,random_state=seed),
                    ExtraTreesClassifier(n_estimators=240,max_depth=12,min_samples_leaf=4,
                     max_features=.72,class_weight='balanced',n_jobs=-1,random_state=seed)]
            for m in models:m.fit(X_train[fields].replace([np.inf,-np.inf],0).fillna(0),y_train)
            group_rows=[];scores=[];labels=[]
            for stem in AUDIT:
                info=AUDIT_BANK[stem];frame=info['features'][fields].replace([np.inf,-np.inf],0).fillna(0)
                probs=np.mean([m.predict_proba(frame)[:,1] for m in models],axis=0) if len(frame) else np.array([])
                indices=[i for i in range(info['n_base']) if probs[i]>=V25_QUALITY_CUTOFF]
                masks=[as_rle(info['masks'][i]) for i in indices]
                gray=cv2.imread(str(TRAIN[stem]),0)
                masks=[real_refine(stem,mask,gray) if probs[i]>=.50 else mask
                       for i,mask in zip(indices,masks)]
                masks=nonoverlap_after_refinement(masks)
                c=physical_counts(stem,masks)
                group_rows.append(dict(stem=stem,day=stem[:8],seed=seed,features=name,
                                 sum_iou=c[0],TP=c[1],FP=c[2],FN=c[3],instances=len(masks)))
                if seed==RANKER_SEEDS[0]:
                    scores.extend(probs.tolist())
                    labels.extend(consensus_quality_labels(stem,[as_rle(q) for q in info['masks']]))
            gf=pd.DataFrame(group_rows)
            counts=gf[['sum_iou','TP','FP','FN']].sum().to_numpy(float)
            FEATURE_RESULTS.append(dict(seed=seed,ablation=name,n_features=len(fields),
                        **metrics(counts),observed_historical_audit=True))
            FEATURE_PRED.extend(group_rows)
            if seed==RANKER_SEEDS[0] and name=='FULL_69':
                np.savez_compressed(OUT/'evidence'/'full_ranker_calibration_scores.npz',
                       scores=np.asarray(scores),labels=np.asarray(labels,dtype=np.uint8))
    feature_results=pd.DataFrame(FEATURE_RESULTS)
    feature_results.to_csv(OUT/'tables'/'Table06_retrained_feature_group_ablation.csv',index=False)
    pd.DataFrame(FEATURE_PRED).to_csv(OUT/'tables'/'Table06_feature_group_per_scene.csv',index=False)
    mean_feature=feature_results.groupby('ablation').PQ.agg(['mean','std','min','max']).reset_index()
    mean_feature.to_csv(OUT/'tables'/'Table06_seed_summary.csv',index=False)
    display(mean_feature)
else:
    print('Feature-group retraining OFF; component removals and all other analyses still run.')

## 8 · Figures: ablation PQ, bootstrap deltas, size recall, annotator agreement, calibration

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

def export_figure(name):
    plt.savefig(OUT/'figures'/f'{name}.png',dpi=220,bbox_inches='tight')
    plt.savefig(OUT/'figures'/f'{name}.pdf',bbox_inches='tight')
    plt.close()

s=summary[summary.partition=='AUDIT_historical'].sort_values('PQ')
plt.figure(figsize=(10,7))
plt.barh(s.method,s.PQ)
plt.xlabel('Historical audit proxy: Panoptic Quality')
plt.title('Identical scenes · fixed learned V25 checkpoints')
plt.xlim(0,max(.50,float(s.PQ.max())+.03))
export_figure('Fig01_V25_component_ablation_PQ')

bs=bootstrap[bootstrap.method!='V25_full_reconstructed'].sort_values('delta_vs_full')
fig,ax=plt.subplots(figsize=(10,7))
ypos=np.arange(len(bs));err=[bs.delta_vs_full-bs.lower_95,bs.upper_95-bs.delta_vs_full]
ax.errorbar(bs.delta_vs_full,ypos,xerr=err,fmt='o',capsize=3)
ax.axvline(0,linewidth=1,linestyle='--',color='gray')
ax.set_yticks(ypos,bs.method)
ax.set_xlabel('Paired PQ difference vs reconstructed V25 (95% day bootstrap interval)')
ax.set_title('Descriptive uncertainty · historically reused audit')
export_figure('Fig02_component_PQ_deltas_CI')

plt.figure(figsize=(8,4))
plt.bar(size_summary.group,size_summary.recall)
plt.ylim(0,1);plt.xticks(rotation=20,ha='right')
plt.ylabel('Matched-instance recall (IoU > 0.5)')
plt.title('Instance size sensitivity · V25 full, historical audit')
export_figure('Fig03_V25_instance_size_recall')

if len(agreement_df):
    plt.figure(figsize=(7,4));plt.hist(agreement_df.PQ,bins=15)
    plt.xlabel('Pairwise annotator PQ');plt.ylabel('Pairs')
    plt.title('Expert annotation disagreement (NOT a performance ceiling)')
    export_figure('Fig04_expert_agreement')

if RUN_RANKER_FEATURE_ABLATIONS:
    f=mean_feature.sort_values('mean')
    plt.figure(figsize=(9,5));plt.barh(f.ablation,f['mean'],xerr=f['std'].fillna(0))
    plt.xlabel('Audit PQ across re-trained quality-ranker seeds')
    plt.title('Feature-family retraining · same historical audit')
    export_figure('Fig05_retrained_ranker_feature_ablation')

cal=OUT/'evidence'/'full_ranker_calibration_scores.npz'
if cal.exists():
    dat=np.load(cal);prob=dat['scores'];y=dat['labels']
    frames=[]
    for low in np.arange(0,1,.1):
        sel=(prob>=low)&(prob<low+.1)
        if sel.sum()>=4:frames.append((float(prob[sel].mean()),float(y[sel].mean()),int(sel.sum())))
    pd.DataFrame(frames,columns=['mean_predicted','observed_positive','n']).to_csv(
        OUT/'tables'/'Table07_ranker_reliability.csv',index=False)
    plt.figure(figsize=(6,5));plt.plot([0,1],[0,1],linestyle='--',color='gray')
    if frames:plt.scatter([v[0] for v in frames],[v[1] for v in frames],s=[max(25,v[2]) for v in frames])
    plt.xlabel('Mean predicted instance-quality probability')
    plt.ylabel('Observed agreement-positive fraction')
    plt.title('Historical audit quality-ranker reliability (new seed-refit)')
    export_figure('Fig06_ranker_reliability')
print('Figure PDFs/PNGs:',len(list((OUT/'figures').glob('*'))))

## 9 · Actual H-alpha image overlays: TP, FP, FN examples (not simulated images)

In [ ]:
# The plotted source images and target masks are from official labeled TRAIN, never hidden test ground truth.
def overlay_image(stem,mask_rles,truth_rles,side=820):
    gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
    assert gray is not None
    bg=cv2.resize(gray,(side,side),interpolation=cv2.INTER_AREA)
    rgb=cv2.cvtColor(bg,cv2.COLOR_GRAY2RGB)
    # Cyan = predicted; yellow = expert annotations; red = FP only; green = TP only.
    pred=cv2.resize(foreground_union(mask_rles),(side,side),interpolation=cv2.INTER_NEAREST)>0
    gt_mask=cv2.resize(foreground_union(truth_rles),(side,side),interpolation=cv2.INTER_NEAREST)>0
    rgb[gt_mask & ~pred]=(245,165,40)
    rgb[pred & ~gt_mask]=(228,70,70)
    rgb[pred & gt_mask]=(90,230,165)
    return rgb

if RUN_ERROR_VISUALIZATIONS:
    ref_rows=audit_scene[audit_scene.method=='V25_full_reconstructed'].copy()
    ref_rows['per_scene_pq']=ref_rows.sum_iou/(ref_rows.TP+.5*(ref_rows.FP+ref_rows.FN)).replace(0,np.nan)
    ranked=ref_rows.sort_values('per_scene_pq')
    selections=list(ranked.head(VISUALIZATION_CASES//2).stem)+list(ranked.tail(VISUALIZATION_CASES-VISUALIZATION_CASES//2).stem)
    for k,stem in enumerate(dict.fromkeys(selections)):
        variants,_=scene_variants(stem,methods=['D2_two_detector_fusion','V25_no_real_mask_refinement','V25_full_reconstructed'])
        expert=gt(stem)[0] if gt(stem) else []
        fig,axes=plt.subplots(1,3,figsize=(16,5))
        for ax,(method,masks) in zip(axes,variants.items()):
            ax.imshow(overlay_image(stem,masks,expert));ax.set_title(method.replace('_',' '),fontsize=9)
            ax.axis('off')
        fig.suptitle(f'{stem} · green overlap · yellow missed GT · red excess prediction',fontsize=11)
        export_figure(f'Fig07_case_{k+1:02d}_{stem}')
print('Qualitative scenes generated:',len(list((OUT/'figures').glob('Fig07_case_*.png'))))

## 10 · Optional test-image reproduction check: original scored V25 vs reconstructed V25

In [ ]:
# This comparison uses ONLY test image pixels + the user's historical scored predictions.
# Regeneration agreement is NOT a public leaderboard measurement.
REGEN_REPORT=[]
if RUN_TEST_REPRODUCTION:
    for stem in tqdm(sorted(TEST),desc='Compare regenerated V25 vs exact public-scored V25 masks'):
        reconstructed=frozen_v25_masks(stem)
        original=CONTROL_BY_STEM.get(stem,[])
        overlap=np.asarray(coco.iou(original,reconstructed,[0]*len(reconstructed)),float) if original and reconstructed else np.zeros((len(original),len(reconstructed)))
        if overlap.size:
            ri,ci=linear_sum_assignment(-overlap)
            matching=overlap[ri,ci]
            above=matching>.95
            near_identical=int(above.sum())
            mean_matched_iou=float(np.mean(matching))
        else:
            near_identical=0;mean_matched_iou=0.
        REGEN_REPORT.append(dict(stem=stem,original_instances=len(original),regenerated_instances=len(reconstructed),
                                 near_identical_count=near_identical,matched_mean_iou=mean_matched_iou))
    regen=pd.DataFrame(REGEN_REPORT)
    regen.to_csv(OUT/'tables'/'Table08_reconstructed_to_historical_public_agreement.csv',index=False)
    print('Test-scene matching diagnostic:',regen.describe(include='all'))
else:
    (OUT/'evidence'/'test_regeneration_not_executed.txt').write_text(
        'RUN_TEST_REPRODUCTION=False. Exact original V25 public-scored CSV SHA verified; independent reconstruction parity not tested.\n')

## 11 · Architecture and computational-cost figures for the technical report

In [ ]:

# This is a conceptual figure of the actual frozen V25 pipeline, not an additional learned method.
fig,ax=plt.subplots(figsize=(13,4.5))
ax.axis('off')
steps=[('GONG Hα 2048px',.055),('Global YOLO11m\n1792px',.23),('Native YOLO11m\n1280px windows',.23),
       ('NMS + mask\nfusion',.42),('V24 feature\nrefiner',.58),('69-feature\nV25 ranker',.72),
       ('Selective real-mask\nrefinement',.87)]
positions=[(.01,.36,.13,.25),(.18,.6,.17,.22),(.18,.1,.17,.22),(.40,.36,.14,.25),
           (.56,.36,.13,.25),(.72,.36,.13,.25),(.86,.36,.13,.25)]
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
for (label,_),(x,y,w,h) in zip(steps,positions):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.007',fill=False,linewidth=1.6))
    ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=9)
for a,b in [(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(5,6)]:
    x,y,w,h=positions[a];xx,yy,ww,hh=positions[b]
    start=(x+w,y+h/2);end=(xx,yy+hh/2)
    ax.add_patch(FancyArrowPatch(start,end,arrowstyle='->',mutation_scale=14,linewidth=1.0))
ax.text(.60,.08,'Curve-specialist candidates supply contextual neighborhood features; baseline winning decoder selects detector candidates only.',
        ha='center',va='center',fontsize=8,wrap=True)
ax.set_xlim(0,1.015);ax.set_ylim(0,1)
export_figure('Fig00_frozen_V25_pipeline_schematic')

param_counts={name:int(sum(p.numel() for p in get_engine(name).model.parameters())) for name in WEIGHTS}
for name,model in [('V24_refiner',refiner_v24),('V25_curve_proposer',curve_net),('V25_real_refiner',refiner_real)]:
    param_counts[name]=int(sum(p.numel() for p in model.parameters()))
resource=pd.DataFrame([dict(component=name,parameters=value,megaparameters=value/1e6) for name,value in param_counts.items()])
resource.to_csv(OUT/'tables'/'Table09_parameter_counts.csv',index=False)
resource_report={'gpu_peak_allocated_gib':round(torch.cuda.max_memory_allocated(GPU_ID)/2**30,3),
  'all_variant_scenes_seconds_total':float(sum(t['seconds_all_variants'] for t in TIMINGS)),
  'mean_all_variants_s_per_scene':float(np.mean([t['seconds_all_variants'] for t in TIMINGS])),
  'parameter_counts':param_counts,
  'caveat':'Timing includes shared predictor caching and component ablations. NOT a pure single-model inference latency benchmark.'}
save_json(OUT/'evidence'/'resource_report.json',resource_report)
print('Parameter counts and measured joint-study computational cost written.')


## 12 · Manuscript-ready summary, limitations ledger, software versions, archive checks

In [ ]:
import platform,importlib.metadata as mdlib,zipfile
from datetime import datetime,timezone

# Submission provenance and result status are intentionally separate from numerical same-scene comparisons.
versions={}
for package in ['torch','ultralytics','numpy','pandas','opencv-python','pycocotools','scipy','scikit-learn','scikit-image','matplotlib','joblib']:
    try:versions[package]=mdlib.version(package)
    except Exception:versions[package]='missing'
checkpoint_files=[V24_REFINER_PATH,V24_QUALITY_PATH,V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH,*WEIGHTS.values()]
checkpoint_manifest=[dict(file=p.name,sha256=sha(p),size_bytes=p.stat().st_size) for p in checkpoint_files]
save_json(OUT/'evidence'/'environment_and_checkpoints.json',dict(
    packages=versions,python=platform.python_version(),gpu=torch.cuda.get_device_name(GPU_ID),
    checkpoint_fingerprints=checkpoint_manifest,seed=SEED,ranker_seeds=RANKER_SEEDS,
    historical_audit_days=len(DAYS),calibration_scenes=len(CAL),audit_scenes=len(AUDIT),
    official_evaluator_parity='UNVERIFIED',test_ground_truth_used=False))

note=f"""# MAGFiLO Final Paper Evidence — V25 verified 0.39 public submission

## Evidence authority
- Historical Kaggle public PQ: **0.39**, user-confirmed for original V25 CSV only.
- Historical CSV SHA256: `{sha(PROVEN)}`; rows: `{len(control_frame)}`.
- Frozen V25 reconstruction local audit PQ: **{float(ctrl):.6f}** (historically reused CAL/AUDIT; NOT Kaggle public PQ).
- Local audit = 51 scenes, {len(DAYS)} acquisition days. Historical calibration = 53 scenes.
- Full pipeline has 2 frozen YOLO11m-seg sources (global 1792px and native 1280 crops), V24 15-feature quality context/refiner, curve candidate U-Net, V25 69-feature ranker (cutoff .38), V25 real-mask refiner.

## Experimental reports
- `tables/Table01_same_scene_component_ablation.csv`: actual component-removal/addition results on identical scenes.
- `tables/Table02_paired_day_bootstrap.csv`: descriptive date-cluster confidence intervals; do not treat as independent-test p-values.
- `tables/Table03_inter_annotator_agreement.csv`: expert versions and agreement; do not call it an upper bound.
- `tables/Table04_instance_size_recall.csv`: per-instance recall by ground-truth size.
- `tables/Table05_overlap_multiplicity.csv`: split/merge heuristics, not physically verified structure claims.
- `tables/Table06_retrained_feature_group_ablation.csv`: separate quality-ranker fitting; seeds: {RANKER_SEEDS if RUN_RANKER_FEATURE_ABLATIONS else 'not executed'}.
- `figures/Fig07_case_*`: genuine annotated solar-disk images, detector vs ranker vs full reconstructed masks.
- `tables/Table08_reconstructed_to_historical_public_agreement.csv`: optional comparison to exact scored CSV; mode executed: {RUN_TEST_REPRODUCTION}.

## Mandatory limitations for manuscript
1. The 104 V20 detector-holdout images have been repeatedly inspected/tuned in V20–V29. The 53/51 split is **not** independent of prior method development.
2. The local pooled PQ scorer is the historical proxy; its parity with the official competition evaluator has not been verified.
3. Training seeds for frozen V20/V24/V25 convolutional backbones are NOT replicated by this notebook. The ranker seed study is NOT full multi-seed network replication.
4. Ablations are intervention-at-inference on already-trained components. They estimate removal effects; full retraining-without-component is a distinct, more expensive causal question.
5. V25 curve masks are not directly accepted by the historical winning decoder; curve candidates can influence ranker neighborhood/context. Explicit `V25_add_curve_candidates` is an experimental alternative, not part of the proven scored pipeline.
6. No claims of V26–V30 public improvements are made. Those experimental methods never supersede the original 0.39 score here.
7. Multi-annotator physical PQ aggregation and macro-Dice must be checked against the organizer's evaluator before submission.
8. Competition report and high-tier independent research paper have different evidential requirements. This notebook serves the former and core experimental analysis for the latter.
"""
(OUT/'evidence'/'MANUSCRIPT_EVIDENCE_README.md').write_text(note)

# Export a paper table without copy-pasting unverified fabricated numbers.
main=summary[summary.partition=='AUDIT_historical'].sort_values('PQ',ascending=False)
main[['method','PQ','SQ','RQ','dice','TP','FP','FN']].to_csv(OUT/'tables'/'PAPER_MAIN_TABLE.csv',index=False)
(OUT/'tables'/'PAPER_MAIN_TABLE.tex').write_text(main[['method','PQ','SQ','RQ','dice','TP','FP','FN']].to_latex(index=False,float_format='%.4f'))

# Archive only deliverables (never zip model weights again: checkpoint SHA manifests record exact weights).
files=sorted(x for x in OUT.rglob('*') if x.is_file() and x.relative_to(OUT).parts[0] in ('tables','figures','evidence'))
ledger=[dict(file=str(x.relative_to(OUT)),sha256=sha(x),size_bytes=x.stat().st_size) for x in files]
save_json(OUT/'evidence'/'file_manifest.json',dict(files=ledger,generated_utc=datetime.now(timezone.utc).isoformat()))
files=sorted(x for x in OUT.rglob('*') if x.is_file() and x.relative_to(OUT).parts[0] in ('tables','figures','evidence'))
ZIP=Path('/kaggle/working/SOLAR_V25_FINAL_PAPER_EVIDENCE.zip')
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as archive:
    for f in tqdm(files,desc='Archive paper figures, metrics and provenance'):
        archive.write(f,str(f.relative_to(OUT)))
with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None,'Evidence archive failed CRC test'
print('COMPLETE:',ZIP,round(ZIP.stat().st_size/1048576,2),'MiB',len(files),'evidence files')
print('Confirmed public 0.39 original CSV fingerprint:',sha(PROVEN)[:24])
print('Frozen V25 local historical-audit PQ:',round(float(ctrl),6))
print('Paper evidence ready; run official organizer scoring separately to verify evaluator parity.')

## Interpretation before writing the report

This experiment directly answers which **existing V25 components** materially influence the *same-scene* reconstructed pipeline, without changing the proven public 0.39 submission. A 4-page competition technical report can use the measured ablation tables, overlays, architecture diagram, and reproducibility manifest. An independent journal/conference claim needs additional independently held-out solar observations, true multiple-seed backbone training, and official evaluator agreement.

**Integrity checks:** all `.pt` files must match the genuine historical weights; the quality ensemble must be the real 69-feature cutoff-0.38 model; the original V25 CSV must match its known SHA prefix and 1,093 rows; test annotations are never read; figure overlays are from official labeled training images only. The output ZIP records every model fingerprint and file hash. No new leaderboard improvement is asserted.